# 四种模式的验证

仅有 pydantic 模式会对不符合要求的 json 格式进行拦截

## 1、Pydantic格式的验证

In [1]:
from pydantic import BaseModel, Field, SecretStr

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_base="http://localhost:8889",
    api_key=SecretStr("...")
)

class MovieModel(BaseModel):
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")


model_with_structure = model.with_structured_output(MovieModel)
response = model_with_structure.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

ValidationError: 2 validation errors for MovieModel
title
  Field required [type=missing, input_value={'title1': '盗梦空间'...诺兰', 'rating': 9.3}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
year
  Field required [type=missing, input_value={'title1': '盗梦空间'...诺兰', 'rating': 9.3}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

## 2、TypedDict格式的验证

In [2]:
from typing_extensions import TypedDict, Annotated

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_base="http://localhost:8889",
    api_key=SecretStr("sk-01323021c664464b8a4187d9444259f7")
)

class MovieDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str,"电影标题"]
    year: Annotated[int,"电影上映年份"]
    director: Annotated[str,"导演"]
    rating: Annotated[float,"电影评分，满分十分"]


structured_model = model.with_structured_output(MovieDict)
response = structured_model.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

{'title1': '盗梦空间', 'year2': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3}
<class 'dict'>


## 3、JSON Schema格式的验证

In [3]:
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_base="http://localhost:8889",
    api_key=SecretStr("sk-01323021c664464b8a4187d9444259f7")
)

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

structured_model = model.with_structured_output(
    json_schema,
    method="json_schema"
)

response = structured_model.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

{'title1': '盗梦空间', 'year2': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3}
<class 'dict'>


## 4、@dataclass格式的验证

In [4]:
from dataclasses import dataclass

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_base="http://localhost:8889",
    api_key=SecretStr("sk-01323021c664464b8a4187d9444259f7")
)

@dataclass
class Movie():
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")


structured_model = model.with_structured_output(Movie)

response = structured_model.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

{'title1': '盗梦空间', 'year2': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3}
<class 'dict'>
